# PCOS Ovarian Ultrasound Analysis

This notebook explores ovarian ultrasound measurements for PCOS classification. It creates a nine-feature Ovarian State Representation (OSR), compares several classifiers, and performs exploratory feature interpretation and clustering.

**Data privacy:** place an authorized `PCOS_data.csv` in the repository root. The file is excluded from Git. Do not export or commit row-level patient data.

This is an academic analysis, not a clinical diagnostic tool. Run all cells with the authorized dataset before reporting results.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.stats import ttest_ind
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report,
                             f1_score, precision_score, recall_score,
                             roc_auc_score, RocCurveDisplay)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load and prepare the data

In [ ]:
data_path = Path("PCOS_data.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Place an authorized copy of PCOS_data.csv in the repository root. "
        "The dataset is not included in this repository."
    )

df = pd.read_csv(data_path)
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)
df = df.loc[:, ~df.columns.str.startswith("Unnamed:")]

target_col = "PCOS (Y/N)"
ovarian_cols = [
    "Follicle No. (L)",
    "Follicle No. (R)",
    "Avg. F size (L) (mm)",
    "Avg. F size (R) (mm)",
    "Endometrium (mm)",
]
required = [target_col, *ovarian_cols]
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f"Required columns are missing: {missing}")

for col in required:
    df[col] = pd.to_numeric(df[col].replace(["?", "??", " "], np.nan), errors="coerce")
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)
print(f"Records with a target label: {len(df)}")
print(df[target_col].value_counts().rename(index={0: "Non-PCOS", 1: "PCOS"}))


## 3. Exploratory comparison

In [ ]:
# Welch's t-test compares group means without assuming equal variances.
eda_rows = []
for col in ovarian_cols:
    pcos = df.loc[df[target_col] == 1, col].dropna()
    non_pcos = df.loc[df[target_col] == 0, col].dropna()
    statistic, p_value = ttest_ind(pcos, non_pcos, equal_var=False, nan_policy="omit")
    eda_rows.append({
        "measure": col,
        "PCOS mean": pcos.mean(),
        "Non-PCOS mean": non_pcos.mean(),
        "p-value": p_value,
    })
pd.DataFrame(eda_rows).sort_values("p-value")


## 4. Engineer ovarian state features

In [ ]:
features = df[ovarian_cols].copy()
features["Total_Follicles"] = features["Follicle No. (L)"] + features["Follicle No. (R)"]
features["Follicle_Imbalance"] = (
    features["Follicle No. (L)"] - features["Follicle No. (R)"]
).abs()
features["Avg_Follicle_Size"] = (
    features["Avg. F size (L) (mm)"] + features["Avg. F size (R) (mm)"]
) / 2
features["Ovary_Density_Index"] = (
    features["Total_Follicles"] / (features["Avg_Follicle_Size"] + 1)
)
feature_names = features.columns.tolist()
X = features
y = df[target_col]


## 5. Split before preprocessing and balancing

In [ ]:
# Hold out untouched observations before fitting imputation, scaling, or SMOTE.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
base_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, max_depth=4, learning_rate=0.05,
        eval_metric="logloss", random_state=RANDOM_STATE
    ),
    "CatBoost": CatBoostClassifier(
        iterations=250, depth=5, learning_rate=0.05,
        verbose=0, random_seed=RANDOM_STATE
    ),
}

def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])

# Imputation, scaling, and SMOTE are refitted inside every training fold.
cv_auc = {}
for name, model in base_models.items():
    scores = cross_val_score(
        make_pipeline(model), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
    )
    cv_auc[name] = scores
    print(f"{name}: CV ROC-AUC = {scores.mean():.3f} +/- {scores.std():.3f}")


## 6. Compare models on the held-out test set

In [ ]:
voter = VotingClassifier(
    estimators=[(name.lower().replace(" ", "_"), model)
                for name, model in base_models.items()],
    voting="soft",
)
models = {name: make_pipeline(model) for name, model in base_models.items()}
models["Soft Voting Ensemble"] = make_pipeline(voter)

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predicted),
        "Precision": precision_score(y_test, predicted, zero_division=0),
        "Recall": recall_score(y_test, predicted, zero_division=0),
        "F1": f1_score(y_test, predicted, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probability),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(results_df.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_title("Held-out ROC curves")
plt.tight_layout()
plt.show()


## 7. Inspect feature contributions with SHAP

In [ ]:
# SHAP is calculated for the fitted Random Forest using held-out observations.
rf_pipeline = models["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
X_test_imputed = rf_pipeline.named_steps["imputer"].transform(X_test)
X_test_scaled = rf_pipeline.named_steps["scaler"].transform(X_test_imputed)
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_scaled)

if isinstance(shap_values, list):
    positive_class_values = shap_values[1]
elif np.asarray(shap_values).ndim == 3:
    positive_class_values = np.asarray(shap_values)[:, :, 1]
else:
    positive_class_values = shap_values

shap.summary_plot(positive_class_values, X_test_scaled,
                  feature_names=feature_names, show=True)


## 8. Explore ovarian measurement patterns

In [ ]:
# Clusters are descriptive groupings of measurements, not clinical subtypes.
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_for_clustering = scaler.fit_transform(imputer.fit_transform(X))
clusterer = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
cluster_labels = clusterer.fit_predict(X_for_clustering)
print(pd.Series(cluster_labels, name="Exploratory cluster").value_counts().sort_index())


## Interpretation note

Use these results as exploratory evidence only. Do not describe the predicted probabilities as clinical risk or a diagnosis. Report evaluation metrics only from a fresh run of this leakage-controlled workflow, and clearly state the dataset and split used.